In [ ]:
!pip install tensorflow
!pip install numpy
!pip install matplotlib
!pip install scikit-learn
!pip install pandas
!pip install kagglehub
!pip install pillow

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import shutil
import zipfile
import kagglehub
from pathlib import Path
import sklearn.model_selection as ms
import pandas as pd

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# 1. DATASET DOWNLOAD
# ------------------------------------------------------------
print("Downloading Face Mask Dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("shiekhburhan/face-mask-dataset")
print(f"Dataset downloaded to: {dataset_path}")
dataset_path = Path(dataset_path)
zip_files = list(dataset_path.glob("*.zip"))
if zip_files:
    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(dataset_path)
            print(f"Extracted: {zip_file.name}")

# Find the actual image directory (first subdirectory that contains images)
img_dir = None
for item in dataset_path.iterdir():
    if item.is_dir():
        subdirs = [d.name.lower() for d in item.iterdir() if d.is_dir()]
        if any(cls in subdirs for cls in ["with_mask", "without_mask", "incorrect_mask"]):
            img_dir = item
            break
if img_dir is None:
    for item in dataset_path.iterdir():
        if item.is_dir() and any(sub.is_dir() for sub in item.iterdir()):
            img_dir = item
            break
if img_dir is None:
    img_dir = dataset_path

print(f"Original image directory: {img_dir}")
#(Kaggle/Colab)
if "/kaggle/input/" in str(img_dir) or "/kaggle/input" in str(img_dir):
    writable_dir = Path("/kaggle/working/FMD_DATASET")
    if not writable_dir.exists():
        print("Copying dataset from read‑only location to writable directory...")
        shutil.copytree(img_dir, writable_dir)
    img_dir = writable_dir
    print(f"Now using writable dataset at: {img_dir}")
else:
    print("Dataset is already in a writable location.")

# ------------------------------------------------------------
# 2. FLATTEN NESTED SUBDIRECTORIES (e.g., complex/, simple/, mmc/, mc/)
# ------------------------------------------------------------
for class_folder in img_dir.iterdir():
    if class_folder.is_dir():
        for subdir in class_folder.iterdir():
            if subdir.is_dir():
                print(f"Flattening: {subdir} -> {class_folder}")
                for img_file in subdir.glob("*.*"):
                    if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
                        dest = class_folder / img_file.name
                        counter = 1
                        while dest.exists():
                            stem = img_file.stem
                            suffix = img_file.suffix
                            dest = class_folder / f"{stem}_{counter}{suffix}"
                            counter += 1
                        shutil.move(str(img_file), str(dest))
                subdir.rmdir()
                print(f"  Removed empty subdir: {subdir}")

print("Flattening complete. Dataset structure is now flat per class.")

# List class directories
class_dirs = [d for d in img_dir.iterdir() if d.is_dir()]
print("Class directories after flattening:", [d.name for d in class_dirs])

# Expected class names
expected_classes = ["with_mask", "without_mask", "incorrect_mask"]
class_names = [c for c in expected_classes if any(c in d.name.lower() for d in class_dirs)]
print("Classes found:", class_names)
print(f"Number of classes: {len(class_names)}")

if len(class_names) != 3:
    raise ValueError(f"Expected 3 classes, found {len(class_names)}. Check folder names.")

# ------------------------------------------------------------
# 3. DATA PREPROCESSING AND SPLIT (70/15/15)
# ------------------------------------------------------------
IMG_SIZE = 128
BATCH_SIZE = 32
SEED = 42
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Collect all file paths and labels
filepaths = []
labels = []
for class_name in class_names:
    class_path = img_dir / class_name
    for img_file in class_path.glob("*.*"):
        if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            filepaths.append(str(img_file))
            labels.append(class_name)

label_to_index = {name: idx for idx, name in enumerate(class_names)}
y = [label_to_index[lbl] for lbl in labels]

# Split: train+temp (85%) and test (15%)
X_train_temp, X_test, y_train_temp, y_test = ms.train_test_split(
    filepaths, y, test_size=TEST_SPLIT, random_state=SEED, stratify=y
)

# Split train_temp into train (70/85) and val (15/85)
val_ratio = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
X_train, X_val, y_train, y_val = ms.train_test_split(
    X_train_temp, y_train_temp, test_size=val_ratio, random_state=SEED, stratify=y_train_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Custom generator for loading images on the fly
def custom_generator(file_list, label_list, batch_size, img_size, augment=False):
    num_samples = len(file_list)
    while True:
        indices = np.random.permutation(num_samples)
        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            batch_indices = indices[start:end]
            batch_files = [file_list[i] for i in batch_indices]
            batch_labels = [label_list[i] for i in batch_indices]

            images = []
            for fpath in batch_files:
                img = tf.keras.utils.load_img(fpath, target_size=img_size)
                img_array = tf.keras.utils.img_to_array(img) / 255.0
                if augment and np.random.rand() > 0.5:
                    img_array = np.fliplr(img_array)
                images.append(img_array)
            X_batch = np.array(images)
            y_batch = tf.keras.utils.to_categorical(batch_labels, num_classes=len(class_names))
            yield X_batch, y_batch

train_gen = custom_generator(X_train, y_train, BATCH_SIZE, (IMG_SIZE, IMG_SIZE), augment=True)
val_gen = custom_generator(X_val, y_val, BATCH_SIZE, (IMG_SIZE, IMG_SIZE), augment=False)
test_gen = custom_generator(X_test, y_test, BATCH_SIZE, (IMG_SIZE, IMG_SIZE), augment=False)

steps_per_epoch = max(1, len(X_train) // BATCH_SIZE)
validation_steps = max(1, len(X_val) // BATCH_SIZE)
test_steps = max(1, len(X_test) // BATCH_SIZE)

test_dataset = tf.data.Dataset.from_generator(
    lambda: custom_generator(X_test, y_test, BATCH_SIZE, (IMG_SIZE, IMG_SIZE), augment=False),
    output_types=(tf.float32, tf.float32),
    output_shapes=(
        tf.TensorShape([None, IMG_SIZE, IMG_SIZE, 3]),
        tf.TensorShape([None, len(class_names)])
    )
).take(test_steps)

num_classes = len(class_names)
print("Class mapping:", label_to_index)

# ------------------------------------------------------------
# 4. ORIGINAL CNN MODEL (3 conv layers)
# ------------------------------------------------------------
def create_cnn_model():
    model = Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = create_cnn_model()
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

# ------------------------------------------------------------
# 5. TRAINING ORIGINAL CNN
# ------------------------------------------------------------
EPOCHS = 20
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_cnn_model.keras', monitor='val_accuracy', save_best_only=True)

print("\nTraining Original CNN...")
start_time = time.time()
history = cnn_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_gen,
    validation_steps=validation_steps,
    callbacks=[early_stop, checkpoint],
    verbose=1
)
total_time = time.time() - start_time
print(f"Total training time: {total_time:.2f} sec, Avg per epoch: {total_time/EPOCHS:.2f} sec")

# Plot accuracy/loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('Original CNN Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Original CNN Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.savefig('accuracy_loss_curves.png', dpi=150)
plt.show()

# Evaluate on test set
test_loss, test_acc = cnn_model.evaluate(test_dataset, steps=test_steps, verbose=0)
print(f"\nOriginal CNN - Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

# Predictions for classification report
y_true, y_pred = [], []
for X_batch, y_batch in test_dataset:
    preds = cnn_model.predict(X_batch, verbose=0)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
print("\nClassification Report (Original CNN):")
print(classification_report(y_true, y_pred, target_names=class_names))

# ------------------------------------------------------------
# 6. SIMPLE CNN MODEL (1 conv layer)
# ------------------------------------------------------------
def create_simple_model():
    model = Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

simple_model = create_simple_model()
simple_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("\nSimple CNN Summary:")
simple_model.summary()

print("\nTraining Simple CNN...")
start_simple = time.time()
simple_history = simple_model.fit(
    train_gen, steps_per_epoch=steps_per_epoch, epochs=EPOCHS,
    validation_data=val_gen, validation_steps=validation_steps, verbose=1
)
simple_time = time.time() - start_simple
simple_test_loss, simple_test_acc = simple_model.evaluate(test_dataset, steps=test_steps, verbose=0)
print(f"Simple CNN - Test Accuracy: {simple_test_acc:.4f}, Time: {simple_time:.2f}s, Params: {simple_model.count_params():,}")

# ------------------------------------------------------------
# 7. RESNET50 TRANSFER LEARNING MODEL
# ------------------------------------------------------------
def create_resnet50_model():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    model = Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

resnet_model = create_resnet50_model()
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("\nResNet50 Model Summary:")
resnet_model.summary()

print("\nTraining ResNet50...")
start_resnet = time.time()
resnet_history = resnet_model.fit(
    train_gen, steps_per_epoch=steps_per_epoch, epochs=10,
    validation_data=val_gen, validation_steps=validation_steps, verbose=1
)
resnet_time = time.time() - start_resnet
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(test_dataset, steps=test_steps, verbose=0)

trainable_params = sum(tf.keras.backend.count_params(w) for w in resnet_model.trainable_weights)
non_trainable_params = sum(tf.keras.backend.count_params(w) for w in resnet_model.non_trainable_weights)
total_resnet_params = trainable_params + non_trainable_params
print(f"ResNet50 - Test Accuracy: {resnet_test_acc:.4f}, Time: {resnet_time:.2f}s, Total Params: {total_resnet_params:,}")

# ------------------------------------------------------------
# 8. MODEL COMPARISON
# ------------------------------------------------------------
comparison = pd.DataFrame({
    'Model': ['Original CNN', 'Simple CNN', 'ResNet50'],
    'Test Accuracy': [test_acc, simple_test_acc, resnet_test_acc],
    'Training Time (s)': [total_time, simple_time, resnet_time],
    'Parameters': [cnn_model.count_params(), simple_model.count_params(), total_resnet_params]
})
print("\n=== Model Comparison ===")
print(comparison.to_string(index=False))

# Bar plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].bar(comparison['Model'], comparison['Test Accuracy'], color=['blue','green','orange'])
axes[0].set_title('Test Accuracy')
axes[0].set_ylim([0,1])
for i,v in enumerate(comparison['Test Accuracy']): axes[0].text(i, v+0.01, f'{v:.3f}', ha='center')
axes[1].bar(comparison['Model'], comparison['Training Time (s)'], color=['blue','green','orange'])
axes[1].set_title('Training Time (s)')
axes[2].bar(comparison['Model'], comparison['Parameters'], color=['blue','green','orange'])
axes[2].set_title('Number of Parameters')
axes[2].ticklabel_format(style='scientific', axis='y')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

# Training history comparison
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
axs[0,0].plot(history.history['accuracy'], label='Train')
axs[0,0].plot(history.history['val_accuracy'], label='Val')
axs[0,0].set_title('Original CNN Accuracy')
axs[0,0].legend(); axs[0,0].grid(True)
axs[0,1].plot(simple_history.history['accuracy'], label='Train')
axs[0,1].plot(simple_history.history['val_accuracy'], label='Val')
axs[0,1].set_title('Simple CNN Accuracy')
axs[0,1].legend(); axs[0,1].grid(True)
axs[1,0].plot(resnet_history.history['accuracy'], label='Train')
axs[1,0].plot(resnet_history.history['val_accuracy'], label='Val')
axs[1,0].set_title('ResNet50 Accuracy')
axs[1,0].legend(); axs[1,0].grid(True)
axs[1,1].plot(history.history['val_loss'], label='Original')
axs[1,1].plot(simple_history.history['val_loss'], label='Simple')
axs[1,1].plot(resnet_history.history['val_loss'], label='ResNet50')
axs[1,1].set_title('Validation Loss Comparison')
axs[1,1].legend(); axs[1,1].grid(True)
plt.tight_layout()
plt.savefig('training_history_comparison.png', dpi=150)
plt.show()

print("\n=== Implementation Complete with ResNet50 Transfer Learning ===")